In [4]:
from pathlib import Path

main_directory = Path("../data/crops_types_yearly_capitanata_03035")

tifs_3035 = {}

for d in main_directory.iterdir():
    if d.is_dir():
        year = d.name
        
        lista_tifs = list(d.rglob("*.tif"))
        
        tifs_3035[year] = [str(tif) for tif in lista_tifs]

In [5]:
from pathlib import Path
import rioxarray

tifs_4326 = {}

for key, value in tifs_3035.items():
    year_file_list = []
    for v in value:
        file_name = v.replace("03035", "4326")
        file_path = Path(file_name)
        
        # Se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # Crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # File GeoTIFF originale (EPSG:3035)
        raster = rioxarray.open_rasterio(v)
        # Riproiezione dell'intero raster in EPSG:4326
        raster_4326 = raster.rio.reproject("EPSG:4326")
        
        raster_4326.rio.to_raster(file_name)
        
        year_file_list.append(file_name)
        
    tifs_4326[key] = year_file_list

In [10]:
from pathlib import Path
import rasterio

points = []

# Prende la lista di file dell'ultimo gruppo
last_file_list = list(tifs_4326.values())[-1]

for tif in last_file_list:
    with rasterio.open(tif) as src:
        bounds = src.bounds

        lon = (bounds.left + bounds.right) / 2
        lat = (bounds.bottom + bounds.top) / 2

        name = Path(tif).stem

        points.append({
            "name": name,
            "lon": round(lon, 6),
            "lat": round(lat, 6)
        })

print(points)

[{'name': 'CLMS_HRLVLCC_CTY_S2021_R10m_E48N21_4326_V01_R00', 'lon': 16.401658, 'lat': 42.250883}, {'name': 'CLMS_HRLVLCC_CTY_S2021_R10m_E48N20_4326_V01_R00', 'lon': 16.307926, 'lat': 41.35077}, {'name': 'CLMS_HRLVLCC_CTY_S2021_R10m_E47N21_4326_V01_R00', 'lon': 15.19595, 'lat': 42.3202}, {'name': 'CLMS_HRLVLCC_CTY_S2021_R10m_E47N20_4326_V01_R00', 'lon': 15.119707, 'lat': 41.418934}]


In [11]:
import json
from pathlib import Path

# Percorso del file JSON nella cartella di lavoro
json_path = Path("../data/points.json")

# Salva i punti (formato: [["nome_file", lon, lat], ...])
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(points, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(points)} punti in {json_path.resolve()}")

✅ Salvati 4 punti in /home/matteo/Developer/github/crop-spatial-classification/data/points.json
